# Tensors, Datasets and Dataloaders
These exercises will be an introduction to Torch and neural networks. So far you should have at least a general idea of what neural networks are. What I now imagine the main question on your mind is how do I begin programming them? While there are many different Python libraries that can be used for the purpose, most of our work is done using PyTorch. PyTorch is a library which is used for machine learning and creating neural networks. This notebook is an exploration of not only how to build a basic neural network but what parts make up PyTorch. This can be spit into 3 main parts:
- What is a Tensor, and how is it different from a Numpy array and a Pandas dataframe.
- What is a dataloader, and how do we make one.
- What is a neural network, and how do modules allow us to implement them.

The reason we use PyTorch rather than its rival Tensorflow is that currently the academic community seem to prefer it. This means no matter what topic we are studying their is a large chance its code is written in Torch.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.nn import Module
import torch.optim as optim
from torch.optim import Optimizer
from torch.utils.data import Dataset, DataLoader, random_split
from torch import Tensor
from torchvision import datasets
from torchvision.transforms import ToTensor
import torchvision.transforms as transforms
from typing import Tuple, List, Dict
import matplotlib.pyplot as plt
import time
import pandas as pd
type LossFN = Union[Module]

### Torch Tensors
If you remember last week we used Numpy arrays. These were similar to Python lists but were faster at the cost of being limited to a single size. And while these are amazing for general mathematical and scientific calulations they aren't quite as fast as we'd actually like for neural networks, especially when it comes to backpropagation and optimisation. Therefore, PyTorch has their own array implementation called tensors that are optimised for neural networks. These can easily be converted between formats as shown below.

In [ ]:
# Converting NDArray -> Tensor
x = np.array([1, 2, 3, 4])
x_tensor = torch.tensor(x)

# Converting List -> Tensor
z = [4, 3, 2, 1]
z_tensor = torch.tensor(z)

# Similar to numpy shape data can be found here as well
print(f"Tensor: {x_tensor}\nWith Shape: {x_tensor.shape}")

The killer feature of Tensors that makes them different from Numpy's arrays is the ability to set the device its calculations will be done on. 

Every computer uses a central processing unit CPU to calculate values and run code. This device is able to do millions of computations every second making operations such as most of the code in this notebook possible however it suffers from 2 big flaws when it comes to computing neural networks. The first is that the CPU is busy running lots of different things to keep your computer working for you. The next is that it is limited by the amount of processing units it has. This low amount makes it slow at doing vector operations such as those found within neural networks. So rather than using a CPU we can use a device dedicated called the graphics processing unit (GPU). A GPU works well for this task as rather than having dozens of processing units that are really powerful, a GPU has thousands of really weak processing units that can quickly compute vector operations. These differences make it much more efficient for us to use a GPU rather than our CPU to train neural networks.

The below code can be used to find what device you can do your calculations on. This can either be your GPU (called CUDA), MPS (for apple computers), or your CPU. This is then followed with an example of putting a tensor onto the device.

In [ ]:
# Find device being used
device = (
    "cuda" # For NVIDIA GPUs
    if torch.cuda.is_available()
    else "mps" # For Apple devies
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device.upper()}")

# Place tensor on device
x_tensor.to(device)

## Dataset
The next part of Tensor flow is the Dataset. From what we have already discussed we know a dataset is a collection of data that can be used to train a ML model. The last question on the *Linear Regression Notebook* saw us iterating through this data as to find a line of best fit. While this solution works, PyTorch provides a way to do this that is a little nicer. Enter PyTorch's `Dataset` abstract base class. 

A `Dataset` in PyTorch is a class which is can be used to easily iterate through data, it just requires 3 methods to be implemented:
- `__init__`, which initialises the dataset usually by loading the data.
- `__len__`, which gets the total length of the dataset.
- `__getitem__`, which gets a bit of data from the dataset, returning it, usually as a tensor.

To give you an idea of what one looks like below is an example for a basic dataset:

In [ ]:
class WeatherDataset(Dataset):
    def __init__(self, path: str):
        """
        Initialise the weather dataset for binary classification.
        """
        dataset = pd.read_csv(path)
        
        # Date processing
        dataset['Date'] = pd.to_datetime(dataset['Date'])
        dataset['Day'] = dataset['Date'].dt.day
        dataset['Month'] = dataset['Date'].dt.month
        dataset['Year'] = dataset['Date'].dt.year
        dataset.drop(['Date', 'Location', 'Evaporation', 'WindGustDir', 'WindDir9am', 'WindDir3pm'], axis=1, inplace=True)
        
        # Fill missing values for numeric columns
        numeric_columns = [
            'MinTemp', 'MaxTemp', 'Rainfall', 'Sunshine', 'WindGustSpeed',
            'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm',
            'Pressure9am', 'Pressure3pm', 'Temp9am', 'Temp3pm'
        ]
        for col in numeric_columns:
            if col in dataset.columns:
                dataset[col] = dataset[col].fillna(dataset[col].mode()[0])
        
        # Map RainToday, RainTomorrow to 1/0
        for x in ("RainToday", "RainTomorrow"):
            dataset[x] = dataset[x].map({"Yes": 1.0, "No": 0.0})
        
        # Remove rows with missing label
        dataset = dataset.dropna()
        dataset = dataset.astype(float)
        
        # Separate features and label
        x = dataset.drop('RainTomorrow', axis=1).values
        y = dataset['RainTomorrow'].values
        self.input = torch.tensor(x, dtype=torch.float64)
        self.label = torch.tensor(y, dtype=torch.bool)

    def __len__(self) -> int:
        return len(self.input)

    def __getitem__(self, idx: int) -> Tuple[Tensor, Tensor]:
        return self.input[idx], self.label[idx]

One of the most important aspects of our dataset class is that we are using tensors. This is important if we are to later use this dataset to train.

This dataset functions somewhat similar to a list with us being able to easily access the data by indexing it. Below is an example of creating the dataset and calling it to see the first element. This should result in a tensor input of with 19 values of the float64 type.

In [ ]:
path = "./data/weatherAUS.csv"
weather_dataset = WeatherDataset(path)
print(f"Here is some example data:\nInput: {weather_dataset[0][0]}\nLabel: {weather_dataset[0][1]}")

### **(Question 1)** Create a dataset for insurance
Last lesson we created a basic machine learning model from the insurance dataset. This dataset attempts to predict the cost of health-care given a few details. We want to wrap the dataset in PyTorch's dataset. An important part is to make sure that it returns items as tensors. Below is a reminder of some code we used last workshop which may help. This uses numpy so be sure to translate this code into tensors.
```python
data = pd.read_csv('data/insurance.csv')
x = data["age"].to_numpy()
y = data["charges"].to_numpy()
```

**Create a basic dataset for the WHOLE insurance that takes as input a path during initialisation and outputs a tensor representing the age of an individual. This should then be turned into a DataLoader for the dataset**. Remember that the `sex` and `smoker` columns need to be encoded. Also remember to use Tensors for `self.input` and `self.label`.

In [ ]:
class InsuranceDataset(Dataset):
    def __init__(self, path: str):
        """Read the dataset and get the input as a tensor, ensuring all columns are encoded"""
        # Your Code Goes Here
        pass
    def __len__(self) -> int:
        """Get the amount of insurance entries"""
        # Your Code Goes Here
        pass
    def __getitem__(self, idx: int) -> Tuple[Tensor, Tensor]:
        """Get an item representing the """
        # Your Code Goes Here
        pass


In [ ]:
# Create tye dataset
path = "./data/insurance.csv"
insurance_data = InsuranceDataset(path)
elem0 = insurance_data[0]
print(f"Example input tensor:\n{elem0[0]}\n\nExample output tensor:\n{elem0[1]}")
assert len(insurance_data[0][0]) == 5, "The length of the input tensor should be 4"
assert insurance_data[0][0].dtype == torch.float64, "Ensure the type of the tensor is float64"

## Dataloader
Once you have created a dataset you can now wrap it in PyTorch's `Dataloader`. A dataloader allows you to iterate easily through a dataset while also giving you the ability to change how it gives you data. In the code above we simply wrap the dataset in it and give it a batch_size and set it to shuffle. 
- `batch_size` allows you to load multiple inputs and labels at the same time. This can speed up training time as it allows you to do more calculations at once. Keep this in mind when creating models.
- `shuffle` randomises the order of the dataset, this is done to mitigate overfitting.

Once we've done this we are ready to train our model. It should output a tensor representing our input, and its associated label. Below is an example of how we can use the dataloader.

In [ ]:
# Wrap the classes in a dataloader, this will allow us to easily retrieve the data
path = "./data/weatherAUS.csv"
weather_dataset = WeatherDataset(path)
weather_dataloader = DataLoader(weather_dataset, batch_size=1, shuffle=True)

# Print dataset
for batch_idx, (x, y) in enumerate(weather_dataloader):
    if batch_idx == 3: break
    print(f"Batch {batch_idx + 1}\nInputs: {x}\nLabels: {y}\n")

### **(Question 2)** Splitting the data
Before training can be done on a dataset needs to be split into two (sometimes three parts). These parts are the **training dataset** which is used by our neural network to train the network, and our **testing dataset** which tests how well our models function against random data. This is usually done with a 80/20 or 70/30 split, although this is fairly dependent on the dataset.

By splitting the dataset we are able to reduce overfitting and see how our accurate our model is. So before we go about training we first need to go about splitting it. This is done by first calculating the training sizes and test sizes. Before then using `random_split` to split the data. After we have done this we can create new dataloaders for both training and testing data.

**Split the dataset into a 80/20 split**. To achieve this keep in mind the function `random_split(dataset, (train_size, test_size))`.

In [ ]:
# Create dataset
path = "./data/insurance.csv"
insurance_data = InsuranceDataset(path)

# Compute train_size, test_size and use random split
# Your Code Goes Here
pass


In [ ]:
# Print the first 3
for batch_idx, (x, y) in enumerate(insurance_data):
    if batch_idx == 3: break
    print(f"Batch {batch_idx + 1}\nInputs: {x}\nLabels: {y}\n")

If you have done everything correct until this point it should produce 3 tensors with 5 elements in as well as an output. From this we are able to train models by using these input tensors and output labels.